In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import getpass
import wrds
import yfinance as yf
from datetime import datetime

In [ ]:
# Cell 2: Connect to WRDS
print("Connecting to WRDS...")
print("Note: Password won't display as you type (this is normal)\n")

username = input("Enter WRDS username: ")
password = getpass.getpass("Enter WRDS password: ")

try:
    db = wrds.Connection(wrds_username=username, wrds_password=password)
    print("\nConnected successfully to WRDS!")
except Exception as e:
    print(f"\nConnection failed: {e}")
    print("Please verify your credentials and try again.")
    raise

In [ ]:
# Cell 3: Define Diverse Company Sample
companies = {
    'AAPL': {'name': 'Apple Inc.', 'sector': 'Technology', 'event': 'Multiple stock splits'},
    'MSFT': {'name': 'Microsoft Corp.', 'sector': 'Technology', 'event': 'None recent'},
    'META': {'name': 'Meta Platforms', 'sector': 'Technology', 'event': 'FB to META rebrand 2021'},
    'JPM':  {'name': 'JPMorgan Chase', 'sector': 'Financial', 'event': 'Acquired Bear Stearns 2008'},
    'BAC':  {'name': 'Bank of America', 'sector': 'Financial', 'event': 'Acquired Merrill Lynch 2008'},
    'XOM':  {'name': 'Exxon Mobil', 'sector': 'Energy/Industrial', 'event': 'Exxon-Mobil merger 1999'},
    'CAT':  {'name': 'Caterpillar Inc.', 'sector': 'Industrial', 'event': 'None recent'},
    'BA':   {'name': 'Boeing Co.', 'sector': 'Aerospace/Industrial', 'event': '737 MAX crisis'},
    'JNJ':  {'name': 'Johnson & Johnson', 'sector': 'Healthcare', 'event': 'Kenvue spinoff 2023'},
    'WMT':  {'name': 'Walmart Inc.', 'sector': 'Retail', 'event': 'None recent'},
    'DIS':  {'name': 'Walt Disney Co.', 'sector': 'Media/Entertainment', 'event': 'Fox acquisition 2019'},
    'T':    {'name': 'AT&T Inc.', 'sector': 'Telecom', 'event': 'WarnerMedia spinoff 2022'}
}

tickers = list(companies.keys())

# Display company selection
company_df = pd.DataFrame.from_dict(companies, orient="index").reset_index()
company_df.columns = ['Ticker', 'Company Name', 'Sector', 'Major Corporate Event']

print("SELECTED COMPANIES FOR ANALYSIS")
print(f"Total companies: {len(companies)}")
print(f"\nSector breakdown:")
print(company_df['Sector'].value_counts())
print("\n")
display(company_df)

In [ ]:
# Cell 4: Fetch CRSP Daily Stock Data (CORRECTED)
print("Fetching CRSP daily stock data...\n")

# CRSP uses PERMNO as the primary identifier
# We join with dsenames to get ticker symbols

crsp_query = """
SELECT a.date, a.permno, a.permco, a.cusip, b.ticker, b.comnam, 
       a.prc, a.openprc, a.bidlo, a.askhi, a.vol, a.ret, a.retx, 
       a.shrout, a.cfacpr, a.cfacshr
FROM crsp.dsf a
LEFT JOIN crsp.dsenames b
ON a.permno = b.permno
AND b.namedt <= a.date
AND a.date <= b.nameendt
WHERE b.ticker IN ('AAPL','MSFT','META','JPM','BAC','XOM',
                   'CAT','BA','JNJ','WMT','DIS','T')
  AND a.date BETWEEN '2020-01-01' AND '2025-12-31'
ORDER BY b.ticker, a.date
"""

crsp_data = db.raw_sql(crsp_query, date_cols=['date'])

print(f"CRSP data retrieved: {crsp_data.shape[0]:,} rows, {crsp_data.shape[1]} columns")
print(f"\nAvailable CRSP fields:")
for i, col in enumerate(crsp_data.columns, 1):
    print(f"  {i}. {col}")

print(f"\nDate range: {crsp_data['date'].min()} to {crsp_data['date'].max()}")
print(f"\nNumber of unique tickers: {crsp_data['ticker'].nunique()}")
print(f"\nSample data:")
display(crsp_data.head(10))

In [ ]:
# Cell 5: Fetch CRSP Delisting Data (CORRECTED)
print("Fetching CRSP delisting data...\n")

delist_query = """
SELECT a.permno, a.dlstdt, a.dlstcd, a.dlret, a.dlretx, a.dlprc,
       b.ticker, b.comnam
FROM crsp.dsedelist a
LEFT JOIN crsp.dsenames b
ON a.permno = b.permno
AND b.namedt <= a.dlstdt
AND a.dlstdt <= b.nameendt
WHERE b.ticker IN ('AAPL','MSFT','META','JPM','BAC','XOM',
                   'CAT','BA','JNJ','WMT','DIS','T')
"""

crsp_delist = db.raw_sql(delist_query, date_cols=['dlstdt'])

print(f"CRSP delisting records: {crsp_delist.shape[0]}")

if len(crsp_delist) > 0:
    print("\nDelisting data found:")
    display(crsp_delist)
else:
    print("\nNo delisting records found for selected companies")
    print("(This is expected - these are all currently active companies)")

In [ ]:
# Cell 6: Fetch Compustat Fundamentals Annual Data
print("Fetching Compustat fundamentals (annual)...\n")

funda_query = """
SELECT gvkey, datadate, tic, conm, cusip,
       sale, at, ni, ebitda, ceq, lt, capx, oancf, ib, csho
FROM comp.funda
WHERE tic IN ('AAPL','MSFT','META','JPM','BAC','XOM',
              'CAT','BA','JNJ','WMT','DIS','T')
  AND datafmt = 'STD'
  AND consol = 'C'
  AND indfmt = 'INDL'
  AND datadate >= '2015-01-01'
ORDER BY tic, datadate
"""

comp_funda = db.raw_sql(funda_query, date_cols=['datadate'])

print(f"Compustat fundamentals: {comp_funda.shape[0]} rows, {comp_funda.shape[1]} columns")
print(f"\nAvailable fundamental fields:")
for i, col in enumerate(comp_funda.columns, 1):
    print(f"  {i}. {col}")

print(f"\nDate range: {comp_funda['datadate'].min()} to {comp_funda['datadate'].max()}")
print(f"\nSample data:")
display(comp_funda.head(10))

In [ ]:
# Cell 7: Fetch Compustat Daily Price Data
print("Fetching Compustat daily price data...\n")

secd_query = """
SELECT gvkey, datadate, tic, prccd, prchd, prcld, 
       cshtrd, ajexdi, trfd
FROM comp.secd
WHERE tic IN ('AAPL','MSFT','META','JPM','BAC','XOM',
              'CAT','BA','JNJ','WMT','DIS','T')
  AND datadate BETWEEN '2020-01-01' AND '2025-12-31'
ORDER BY tic, datadate
"""

comp_secd = db.raw_sql(secd_query, date_cols=['datadate'])

print(f"Compustat daily prices: {comp_secd.shape[0]:,} rows, {comp_secd.shape[1]} columns")
print(f"\nAvailable price fields:")
for i, col in enumerate(comp_secd.columns, 1):
    print(f"  {i}. {col}")

print(f"\nDate range: {comp_secd['datadate'].min()} to {comp_secd['datadate'].max()}")
print(f"\nSample data:")
display(comp_secd.head(10))

In [ ]:
# Cell 8: Fetch Yahoo Finance Data
print("Fetching Yahoo Finance data...\n")

yahoo_data_dict = {}
yahoo_sample_data = []

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period='5y')
        dividends = stock.dividends
        splits = stock.splits
        
        yahoo_data_dict[ticker] = {
            'price_fields': list(hist.columns),
            'data_shape': hist.shape,
            'date_range': (hist.index.min(), hist.index.max()),
            'has_dividends': len(dividends) > 0,
            'has_splits': len(splits) > 0,
            'num_dividends': len(dividends),
            'num_splits': len(splits)
        }
        
        yahoo_sample_data.append({
            'Ticker': ticker,
            'Days': hist.shape[0],
            'Start': hist.index.min().date(),
            'End': hist.index.max().date(),
            'Dividends': len(dividends),
            'Splits': len(splits)
        })
        
        print(f"  {ticker}: {hist.shape[0]} days, {len(dividends)} dividends, {len(splits)} splits")
    except Exception as e:
        print(f"  {ticker}: Failed - {e}")

yahoo_summary = pd.DataFrame(yahoo_sample_data)
print("\nYahoo Finance data summary:")
display(yahoo_summary)

In [ ]:
# Cell 9: Create Feature Comparison Table
feature_comparison = pd.DataFrame({
    'Feature': [
        'Open Price', 'High Price', 'Low Price', 'Close Price', 'Adjusted Close',
        'Trading Volume', 'Daily Return', 'Return ex-Dividends', 'Delisting Return',
        'Shares Outstanding', 'Price Adjustment Factor', 'Delisting Code',
        'Permanent Security ID', 'Permanent Company ID', 'Global Company ID',
        'Sales/Revenue', 'Total Assets', 'Net Income', 'EBITDA',
        'Common Equity', 'Operating Cash Flow', 'Capital Expenditures'
    ],
    'Yahoo Finance': [
        'Open', 'High', 'Low', 'Close', 'Adj Close',
        'Volume', 'Calculate', 'N/A', 'N/A',
        'Via .info', 'N/A', 'N/A',
        'Ticker only', 'N/A', 'N/A',
        'Via financials', 'Via financials', 'Via financials', 'Via financials',
        'Via financials', 'Via financials', 'Via financials'
    ],
    'CRSP': [
        'OPENPRC', 'ASKHI', 'BIDLO', 'PRC', 'Adjusted PRC',
        'VOL', 'RET', 'RETX', 'DLRET',
        'SHROUT', 'CFACPR', 'DLSTCD',
        'PERMNO', 'PERMCO', 'N/A',
        'N/A', 'N/A', 'N/A', 'N/A',
        'N/A', 'N/A', 'N/A'
    ],
    'Compustat': [
        'N/A', 'prchd', 'prcld', 'prccd', 'Via ajexdi',
        'cshtrd', 'Calculate', 'N/A', 'N/A',
        'csho', 'ajexdi', 'N/A',
        'N/A', 'N/A', 'GVKEY',
        'sale', 'at', 'ni', 'ebitda',
        'ceq', 'oancf', 'capx'
    ]
})

print("FEATURE COMPARISON TABLE\n")
display(feature_comparison)

feature_comparison.to_csv('feature_comparison_complete.csv', index=False)
print("\nSaved to: feature_comparison_complete.csv")

In [ ]:
# Cell 10: Create Data Availability Summary
summary_data = []

for ticker in tickers:
    crsp_count = len(crsp_data[crsp_data['ticker'] == ticker])
    comp_count = len(comp_secd[comp_secd['tic'] == ticker])
    yahoo_count = yahoo_data_dict.get(ticker, {}).get('data_shape', (0,))[0]
    
    summary_data.append({
        'Ticker': ticker,
        'Company': companies[ticker]['name'],
        'Sector': companies[ticker]['sector'],
        'Yahoo Days': yahoo_count,
        'CRSP Days': crsp_count,
        'Compustat Days': comp_count,
        'Corporate Event': companies[ticker]['event']
    })

summary_df = pd.DataFrame(summary_data)

print("DATA AVAILABILITY SUMMARY BY TICKER\n")
display(summary_df)

summary_df.to_csv('data_availability_summary.csv', index=False)
print("\nSaved to: data_availability_summary.csv")

In [ ]:
# Cell 11: Save All Collected Data to CSV
print("Saving collected data to CSV files...\n")

# Save CRSP data
crsp_data.to_csv('crsp_daily_data.csv', index=False)
print(f"Saved CRSP daily data: {crsp_data.shape[0]:,} rows")

crsp_delist.to_csv('crsp_delisting_data.csv', index=False)
print(f"Saved CRSP delisting data: {crsp_delist.shape[0]} rows")

# Save Compustat data
comp_funda.to_csv('compustat_fundamentals.csv', index=False)
print(f"Saved Compustat fundamentals: {comp_funda.shape[0]} rows")

comp_secd.to_csv('compustat_daily_prices.csv', index=False)
print(f"Saved Compustat daily prices: {comp_secd.shape[0]:,} rows")

print("\nAll data saved successfully!")

In [ ]:
# Cell 13: Generate Summary Report as Markdown
summary_markdown = f"""# PART I - DATA COLLECTION SUMMARY

## Overview
This report summarizes the data collection process for Part I of the comparative study analyzing financial data from Yahoo Finance, CRSP, and Compustat.

## Files Created

The following CSV files have been generated and saved:

1. **feature_comparison_complete.csv** - Comprehensive comparison of data fields across all three sources
2. **data_availability_summary.csv** - Summary of data coverage by ticker and source
3. **crsp_daily_data.csv** - CRSP daily stock price and volume data
4. **crsp_delisting_data.csv** - CRSP delisting information
5. **compustat_fundamentals.csv** - Compustat annual fundamental data
6. **compustat_daily_prices.csv** - Compustat daily price data

## Data Statistics

### Companies Analyzed
- **Total companies:** {len(companies)}
- **Sectors represented:** {company_df['Sector'].nunique()}
- **Companies with major corporate events:** {sum(1 for c in companies.values() if c['event'] != 'None recent')}

### Data Volume
- **CRSP records:** {crsp_data.shape[0]:,} daily observations
- **Compustat fundamental records:** {comp_funda.shape[0]} annual observations
- **Compustat price records:** {comp_secd.shape[0]:,} daily observations

### Date Coverage
- **CRSP date range:** {crsp_data['date'].min().date()} to {crsp_data['date'].max().date()}
- **Compustat fundamentals range:** {comp_funda['datadate'].min().date()} to {comp_funda['datadate'].max().date()}
- **Compustat prices range:** {comp_secd['datadate'].min().date()} to {comp_secd['datadate'].max().date()}

## Key Findings

### Common Features Across All Sources
All three data providers offer basic price and volume information:
- Open, High, Low, Close prices
- Trading volume
- Historical price data

### Unique Features by Source

#### Yahoo Finance
- Analyst recommendations and estimates
- Institutional holdings data
- Easy accessibility for retail users
- Real-time and historical dividends/splits

#### CRSP
- **PERMNO/PERMCO:** Permanent security and company identifiers
- **Delisting returns (DLRET):** Critical for avoiding survivorship bias
- **Return calculations (RET, RETX):** Pre-calculated returns with and without dividends
- **Adjustment factors (CFACPR, CFACSHR):** Precise historical price reconstruction

#### Compustat
- **GVKEY:** Global company identifier
- **Comprehensive fundamentals:** Complete income statements and balance sheets
- **Fundamental data fields:** sale, at, ni, ebitda, ceq, lt, capx, oancf
- **Corporate structure data:** Detailed financial statement items

## Company Selection Rationale

The selected companies represent diverse sectors and include firms that have experienced major corporate events:

### Technology Sector
- Apple (AAPL) - Multiple stock splits
- Microsoft (MSFT) - Stable technology leader
- Meta (META) - Ticker change from FB to META in 2021

### Financial Sector
- JPMorgan Chase (JPM) - Acquired Bear Stearns in 2008
- Bank of America (BAC) - Acquired Merrill Lynch in 2008

### Industrial/Energy Sector
- Exxon Mobil (XOM) - Result of Exxon-Mobil merger in 1999
- Caterpillar (CAT) - Traditional industrial company
- Boeing (BA) - Experienced 737 MAX crisis

### Other Sectors
- Johnson & Johnson (JNJ) - Healthcare, Kenvue spinoff in 2023
- Walmart (WMT) - Retail sector
- Walt Disney (DIS) - Media/Entertainment, Fox acquisition in 2019
- AT&T (T) - Telecommunications, WarnerMedia spinoff in 2022

## Next Steps

### Immediate Tasks
1. Review the feature comparison table to understand field availability across sources
2. Examine data coverage patterns for each company
3. Verify data quality and identify any missing values or anomalies

### Part II: Historical Data Integrity & Survivorship Bias
The next phase will focus on:
- Corporate name and ticker changes (e.g., FB to META)
- Delisting and bankruptcy data handling
- Survivorship bias analysis and mitigation strategies
- CRSP delisting return calculations

### Part III: Price Adjustments & Corporate Actions
Subsequent analysis will cover:
- Stock split adjustments
- Dividend adjustments
- Comparison of adjusted vs. raw prices
- Impact on return calculations

### Part IV: Entity Identification & Data Linkage
Final data preparation will include:
- PERMNO, PERMCO, GVKEY, and CUSIP identifier analysis
- Ticker reuse problems
- Linking CRSP price data with Compustat fundamentals

## Data Quality Notes

### Potential Issues to Investigate
1. **META ticker:** Data before 2021 should appear under FB ticker
2. **Missing values:** Check for gaps in CRSP or Compustat coverage
3. **Date alignment:** Ensure proper matching across sources for same trading days
4. **Corporate actions:** Verify adjustment factors are applied consistently

### Validation Checks Recommended
- Compare adjusted close prices across Yahoo Finance and CRSP
- Verify fundamental data completeness for all companies
- Check for outliers or data quality issues in volume figures
- Confirm delisting data completeness

## Conclusion

Part I successfully collected comprehensive data from three major financial data providers. The diverse company selection includes firms across multiple sectors with various corporate events, providing a robust foundation for analyzing data provider differences and their impact on financial analysis.

The collected datasets are ready for detailed comparative analysis in subsequent project phases.

---

**Report Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Analysis Period:** 2020-01-01 to 2025-12-31
**Number of Companies:** {len(companies)}
**Total Records Collected:** {crsp_data.shape[0] + comp_funda.shape[0] + comp_secd.shape[0]:,}
"""

# Save the markdown report
with open('Part_I_Summary_Report.md', 'w') as f:
    f.write(summary_markdown)

print("Summary report generated and saved to: Part_I_Summary_Report.md")
print("\nYou can now include this markdown file in your project report.")
print("\nTo view the report, open Part_I_Summary_Report.md in any markdown viewer or text editor.")

In [ ]:
# Cell 14: Compare Common Field Values Across Data Sources (CORRECTED)
print("COMPARING COMMON FIELDS ACROSS DATA SOURCES\n")

# First, find a date that exists in all three sources for AAPL
sample_ticker = 'AAPL'

# Get dates available in CRSP for AAPL
crsp_aapl = crsp_data[crsp_data['ticker'] == sample_ticker].copy()
crsp_dates = crsp_aapl['date'].unique()

# Find a recent date
if len(crsp_dates) > 100:
    sample_date = pd.to_datetime(crsp_dates[-100])
else:
    sample_date = pd.to_datetime(crsp_dates[0])

sample_date_str = sample_date.strftime('%Y-%m-%d')

print(f"Sample Comparison: {sample_ticker} on {sample_date_str}\n")

# Get Yahoo Finance data for comparison
import yfinance as yf
yahoo_stock = yf.Ticker(sample_ticker)
start_date = (sample_date - pd.Timedelta(days=5)).strftime('%Y-%m-%d')
end_date = (sample_date + pd.Timedelta(days=5)).strftime('%Y-%m-%d')
yahoo_hist = yahoo_stock.history(start=start_date, end=end_date)

# Filter data for the sample date
yahoo_sample = yahoo_hist[yahoo_hist.index.date == sample_date.date()]
crsp_sample = crsp_data[(crsp_data['ticker'] == sample_ticker) & 
                        (crsp_data['date'] == sample_date)]
comp_sample = comp_secd[(comp_secd['tic'] == sample_ticker) & 
                        (comp_secd['datadate'] == sample_date)]

# Create comparison dataframe
if len(yahoo_sample) > 0 and len(crsp_sample) > 0:
    comparison = pd.DataFrame({
        'Field': ['Open', 'High', 'Low', 'Close', 'Volume'],
        'Yahoo Finance': [
            round(yahoo_sample['Open'].values[0], 2) if len(yahoo_sample) > 0 else 'N/A',
            round(yahoo_sample['High'].values[0], 2) if len(yahoo_sample) > 0 else 'N/A',
            round(yahoo_sample['Low'].values[0], 2) if len(yahoo_sample) > 0 else 'N/A',
            round(yahoo_sample['Close'].values[0], 2) if len(yahoo_sample) > 0 else 'N/A',
            int(yahoo_sample['Volume'].values[0]) if len(yahoo_sample) > 0 else 'N/A'
        ],
        'CRSP': [
            round(float(crsp_sample['openprc'].values[0]), 2) if crsp_sample['openprc'].values[0] > 0 else 'N/A',
            round(float(crsp_sample['askhi'].values[0]), 2) if crsp_sample['askhi'].values[0] > 0 else 'N/A',
            round(float(crsp_sample['bidlo'].values[0]), 2) if crsp_sample['bidlo'].values[0] > 0 else 'N/A',
            round(abs(float(crsp_sample['prc'].values[0])), 2),
            int(crsp_sample['vol'].values[0]) if crsp_sample['vol'].values[0] > 0 else 'N/A'
        ],
        'Compustat': [
            'N/A',
            round(float(comp_sample['prchd'].values[0]), 2) if len(comp_sample) > 0 and comp_sample['prchd'].values[0] > 0 else 'N/A',
            round(float(comp_sample['prcld'].values[0]), 2) if len(comp_sample) > 0 and comp_sample['prcld'].values[0] > 0 else 'N/A',
            round(float(comp_sample['prccd'].values[0]), 2) if len(comp_sample) > 0 and comp_sample['prccd'].values[0] > 0 else 'N/A',
            int(comp_sample['cshtrd'].values[0]) if len(comp_sample) > 0 and comp_sample['cshtrd'].values[0] > 0 else 'N/A'
        ]
    })
    
    print("Single Day Comparison:")
    display(comparison)
    comparison.to_csv('/home/aditya/FE511_Project/single_day_comparison.csv', index=False)
    print("\nSaved to: /home/aditya/FE511_Project/single_day_comparison.csv")
else:
    print(f"Data not available for all sources on {sample_date_str}")

print("\n")

In [ ]:
# Cell 15: Comprehensive Value Comparison Across Multiple Days (CORRECTED)
print("MULTI-DAY VALUE COMPARISON\n")
print("Note: Using UNADJUSTED prices for fair comparison\n")

# Get overlapping dates between CRSP and Compustat
crsp_aapl = crsp_data[crsp_data['ticker'] == 'AAPL'].copy()
comp_aapl = comp_secd[comp_secd['tic'] == 'AAPL'].copy()

# Find common dates
common_dates = set(crsp_aapl['date'].dt.date).intersection(set(comp_aapl['datadate'].dt.date))
comparison_dates = sorted(list(common_dates))[-20:]  # Last 20 common dates

# Get Yahoo data for the date range
if len(comparison_dates) > 0:
    start_date = comparison_dates[0]
    end_date = comparison_dates[-1]
    
    yahoo_stock = yf.Ticker('AAPL')
    yahoo_hist = yahoo_stock.history(start=start_date, end=end_date)
    
    comparison_results = []
    
    for date in comparison_dates[:10]:  # Compare first 10 dates
        # Get Yahoo data (unadjusted close)
        yahoo_day = yahoo_hist[yahoo_hist.index.date == date]
        yahoo_close = yahoo_day['Close'].values[0] if len(yahoo_day) > 0 else None
        
        # Get CRSP data (unadjusted - use raw prc)
        crsp_day = crsp_aapl[crsp_aapl['date'].dt.date == date]
        crsp_close = abs(float(crsp_day['prc'].values[0])) if len(crsp_day) > 0 else None
        
        # Get Compustat data
        comp_day = comp_aapl[comp_aapl['datadate'].dt.date == date]
        comp_close = float(comp_day['prccd'].values[0]) if len(comp_day) > 0 else None
        
        if yahoo_close and crsp_close and comp_close:
            comparison_results.append({
                'Date': str(date),
                'Yahoo_Close': round(yahoo_close, 2),
                'CRSP_Close': round(crsp_close, 2),
                'Compustat_Close': round(comp_close, 2),
                'Yahoo_vs_CRSP_Diff': round(abs(yahoo_close - crsp_close), 4),
                'Yahoo_vs_Comp_Diff': round(abs(yahoo_close - comp_close), 4),
                'CRSP_vs_Comp_Diff': round(abs(crsp_close - comp_close), 4)
            })
    
    if len(comparison_results) > 0:
        comparison_df = pd.DataFrame(comparison_results)
        print("Closing Price Comparison Across Sources:")
        display(comparison_df)
        
        # Calculate statistics
        print("\nDifference Statistics:")
        print(f"Average Yahoo vs CRSP difference: ${comparison_df['Yahoo_vs_CRSP_Diff'].mean():.4f}")
        print(f"Average Yahoo vs Compustat difference: ${comparison_df['Yahoo_vs_Comp_Diff'].mean():.4f}")
        print(f"Average CRSP vs Compustat difference: ${comparison_df['CRSP_vs_Comp_Diff'].mean():.4f}")
        print(f"\nMax difference: ${comparison_df[['Yahoo_vs_CRSP_Diff', 'Yahoo_vs_Comp_Diff', 'CRSP_vs_Comp_Diff']].max().max():.4f}")
        
        comparison_df.to_csv('/home/aditya/FE511_Project/price_comparison_analysis.csv', index=False)
        print("\nSaved to: /home/aditya/FE511_Project/price_comparison_analysis.csv")
    else:
        print("No overlapping data found for comparison")
else:
    print("No common dates found between data sources")

In [ ]:
# Cell 16: Volume Comparison Analysis (CORRECTED)
print("VOLUME COMPARISON ANALYSIS\n")

# Use the same dates from price comparison
if len(comparison_dates) > 0:
    volume_comparison = []
    
    for date in comparison_dates[:10]:  # First 10 dates
        # Yahoo volume
        yahoo_day = yahoo_hist[yahoo_hist.index.date == date]
        yahoo_vol = int(yahoo_day['Volume'].values[0]) if len(yahoo_day) > 0 else None
        
        # CRSP volume
        crsp_day = crsp_aapl[crsp_aapl['date'].dt.date == date]
        crsp_vol = int(crsp_day['vol'].values[0]) if len(crsp_day) > 0 and crsp_day['vol'].values[0] > 0 else None
        
        # Compustat volume
        comp_day = comp_aapl[comp_aapl['datadate'].dt.date == date]
        comp_vol = int(comp_day['cshtrd'].values[0]) if len(comp_day) > 0 and comp_day['cshtrd'].values[0] > 0 else None
        
        if yahoo_vol and crsp_vol:
            volume_comparison.append({
                'Date': str(date),
                'Yahoo_Volume': yahoo_vol,
                'CRSP_Volume': crsp_vol,
                'Compustat_Volume': comp_vol if comp_vol else 'N/A',
                'Match_Yahoo_CRSP': yahoo_vol == crsp_vol,
                'Percent_Diff': round(abs(yahoo_vol - crsp_vol) / yahoo_vol * 100, 2) if crsp_vol else 0
            })
    
    if len(volume_comparison) > 0:
        volume_df = pd.DataFrame(volume_comparison)
        print("Volume Data Comparison:")
        display(volume_df)
        
        # Calculate match statistics
        match_count = volume_df['Match_Yahoo_CRSP'].sum()
        total_count = len(volume_df)
        
        print(f"\nVolume Match Statistics:")
        print(f"Exact matches: {match_count} out of {total_count} ({match_count/total_count*100:.1f}%)")
        print(f"Average percentage difference: {volume_df['Percent_Diff'].mean():.2f}%")
        print(f"Max percentage difference: {volume_df['Percent_Diff'].max():.2f}%")
        
        volume_df.to_csv('/home/aditya/FE511_Project/volume_comparison_analysis.csv', index=False)
        print("\nSaved to: /home/aditya/FE511_Project/volume_comparison_analysis.csv")
    else:
        print("No volume data available for comparison")
else:
    print("No dates available for volume comparison")

In [ ]:
# Cell 19: Generate Analysis Summary Based on Results
analysis_summary = """# Common Features Analysis - Key Findings

## Executive Summary

Based on comprehensive comparison of AAPL data from Yahoo Finance, CRSP, and Compustat (December 2024), we observe high agreement in data values with systematic differences explained by adjustment methodologies.

## Price Data Findings

### Single Day Analysis (August 9, 2024)

| Field | Yahoo Finance | CRSP | Compustat |
|-------|--------------|------|-----------|
| Open | $210.67 | $212.10 | N/A |
| High | $215.32 | $216.78 | $216.78 |
| Low | $210.54 | $211.97 | $211.97 |
| Close | $214.78 | $216.24 | $216.24 |

**Key Observation:** Yahoo Finance prices are consistently **~$1.46 lower** than CRSP/Compustat

### Multi-Day Analysis (December 2024)

- **Average Yahoo vs CRSP difference:** $1.11
- **Average Yahoo vs Compustat difference:** $1.11  
- **Average CRSP vs Compustat difference:** $0.00

**Key Observation:** CRSP and Compustat prices **match exactly**, while Yahoo Finance differs by a consistent amount

## Are Values Identical?

### Answer: **NO for prices, NEARLY YES for volumes**

#### Price Data
- **CRSP and Compustat:** Identical (0.00 difference)
- **Yahoo Finance vs others:** Systematic difference of ~$1-1.50

#### Volume Data
- **Exact matches:** 0 out of 10 (0%)
- **Average difference:** 0.82%
- **Range of differences:** 0.42% to 1.41%

## Why Do Values Differ?

### Price Differences Explanation

The consistent ~$1.11 difference between Yahoo Finance and CRSP/Compustat suggests:

1. **Corporate Action Adjustment Timing**
   - CRSP/Compustat: Applied recent adjustment factor (likely dividend or split)
   - Yahoo Finance: May not have applied the same adjustment or applied it differently
   - The ratio (242.65/241.56 = 1.0045) suggests a small adjustment factor

2. **Adjustment Methodology**
   - CRSP uses cumulative adjustment factors (CFACPR) for academic precision
   - Compustat mirrors CRSP methodology for consistency
   - Yahoo Finance may use different adjustment calculation or timing

3. **Data Update Timing**
   - CRSP/Compustat: End-of-day consolidated data with all adjustments
   - Yahoo Finance: Real-time feed that may lag on corporate action processing

### Volume Differences Explanation

Minor volume differences (< 1.5%) occur due to:

1. **Exchange Reporting**
   - Yahoo: Consolidated tape (all exchanges)
   - CRSP: Primary exchange listing
   - Compustat: S&P Global consolidated feed

2. **After-Hours Trading**
   - Different policies on including extended hours volume
   - Regular session vs total volume definitions

3. **Trade Reporting**
   - Late reports or corrections handled differently
   - Block trade reporting variations

4. **Data Rounding**
   - Different precision in volume aggregation

## Critical Insight: CRSP and Compustat Perfect Agreement

The **zero difference** between CRSP and Compustat is highly significant:

- Indicates both use **identical primary data sources**
- Both apply **same adjustment methodologies**
- Confirms **professional-grade data consistency**
- Makes them interchangeable for price/volume research

## Practical Implications

### For Researchers
1. **Use CRSP or Compustat for academic work** - they match exactly
2. **Yahoo Finance may introduce $1+ errors** if recent corporate actions occurred
3. **Always verify adjustment factors** when combining sources

### For Return Calculations
1. The $1.11 difference on $242 stock = **0.46% error** in single-day calculations
2. Compounds over time in return series
3. Critical for precise performance measurement

### For Data Selection
- **Academic research:** CRSP (gold standard)
- **Fundamental + price analysis:** Compustat (includes financials)
- **Quick prototypes:** Yahoo Finance (free, but verify adjustments)
- **Professional backtesting:** CRSP or Compustat (identical, bias-free)

## Volume Data Assessment

Despite 0% exact matches:
- **0.82% average difference is negligible** for most analyses
- Represents ~300,000 shares difference on 38M daily volume
- **Acceptable for all practical purposes**
- Likely due to reporting timing, not data quality issues

## Conclusion

### Common Features Summary

| Feature | Present in All? | Values Identical? | Usability |
|---------|----------------|-------------------|-----------|
| Close Price | Yes | CRSP=Comp, Yahoo differs | Verify adjustments |
| High Price | Yes | CRSP=Comp, Yahoo differs | Verify adjustments |
| Low Price | Yes | CRSP=Comp, Yahoo differs | Verify adjustments |
| Volume | Yes | Nearly identical (<1% diff) | Excellent agreement |

### Key Takeaway

While all three sources provide common price and volume fields:
- **CRSP and Compustat are interchangeable** (perfect agreement)
- **Yahoo Finance requires careful adjustment verification**
- **Volume data is highly consistent** across all sources (<1% variance)
- **Choice of data source matters** for precise return calculations

The systematic nature of differences (consistent $1.11) rather than random variations confirms these are **methodological differences**, not data quality issues.

---

**Analysis Date:** December 2024  
**Sample Stock:** AAPL (Apple Inc.)  
**Sample Size:** 10 trading days  
**Sources:** Yahoo Finance, CRSP (via WRDS), Compustat (via WRDS)
"""

# Save the analysis summary
with open('/home/aditya/FE511_Project/Common_Features_Analysis_Summary.md', 'w') as f:
    f.write(analysis_summary)

print("Detailed analysis summary saved to:")
print("/home/aditya/FE511_Project/Common_Features_Analysis_Summary.md")
print("\nThis document provides a comprehensive answer to:")
print("  - Are common field values identical?")
print("  - If not, why do they differ?")
print("  - What are the practical implications?")

In [ ]:
# NEW Cell for Part I: Generate WHY Questions Explanation Document
print("Generating 'Why Questions' explanation document...\n")

why_questions_doc = """# Part I: Why Questions - Data Provider Feature Differences Explained

## Overview

This document explains why different data providers have different features, based on their target audience, business model, and core purpose.

---

## Q1: Why doesn't CRSP provide intraday data?

### Answer

CRSP (Center for Research in Security Prices) is designed specifically for **academic research**, not day trading or real-time market analysis.

**Key Reasons:**

1. **Academic Research Focus**
   - Academic studies focus on end-of-day (EOD) pricing for statistical validity
   - Daily returns calculated consistently at market close eliminate intraday noise
   - Long-term research (months/years) does not require minute-by-minute data

2. **Statistical Validity**
   - EOD prices represent final consensus value for the day
   - Intraday data contains microstructure noise (bid-ask bounce, temporary imbalances)
   - Research on returns, volatility, correlations uses daily or monthly data

3. **Data Management**
   - Storing more than 100 years of intraday data would be massive
   - CRSP maintains data since 1926; intraday would be terabytes
   - Processing and adjustment complexity would increase substantially

4. **Not the Mission**
   - CRSP's mission is to provide accurate, adjusted historical prices for research
   - Real-time trading data is covered by databases such as TAQ (Trade and Quote)
   - CRSP focuses on historical accuracy, not trading infrastructure

**Bottom Line:** If intraday data is needed, use TAQ or similar databases. CRSP serves a different purpose.

---

## Q2: Why does Yahoo Finance have analyst recommendations and institutional holdings?

### Answer

Yahoo Finance targets **retail investors** making current investment decisions, not academic researchers studying historical patterns.

**Key Reasons:**

1. **Retail Investor Audience**
   - Retail investors ask: "What should I buy now?"
   - Academic researchers ask: "What patterns existed historically?"
   - These are fundamentally different use cases

2. **Investment Decision Support**
   - Analyst ratings (Buy/Sell/Hold) help non-professionals make decisions
   - Institutional holdings show large investor positions
   - Earnings estimates provide forward-looking guidance
   - These features have little or no value for historical backtesting

3. **Business Model**
   - Yahoo Finance is **free** and **ad-supported**
   - It needs daily users to generate advertising revenue
   - Features that drive engagement include recommendations, real-time quotes, and news
   - Historical research data does not drive traffic

4. **User Engagement**
   - Retail investors check Yahoo daily or weekly
   - Academic researchers run large queries infrequently
   - Recommendations and holdings change frequently, bringing users back
   - This repeat traffic is valuable for advertising

**Bottom Line:** Yahoo optimizes for user engagement and current trading, not historical research accuracy.

---

## Q3: Why does Yahoo Finance not have PERMNO or delisting returns?

### Answer

Yahoo Finance serves **current market participants**, not historical researchers conducting backtests.

**Key Reasons:**

1. **No Academic Research Use Case**
   - PERMNO is an academic construct created by CRSP
   - Retail traders do not need permanent identifiers; they trade current tickers
   - "What is AAPL worth today?" does not require PERMNO
   - "How did this portfolio perform from 1990 to 2020?" does require a permanent identifier

2. **Current Trading Focus**
   - Yahoo users want: "What is META trading at right now?"
   - They do not ask: "What was Facebook's PERMNO in 2015?"
   - The current ticker is sufficient for current trading

3. **Delisting Returns and Revenue**
   - Delisting returns matter for backtest accuracy
   - Most Yahoo users are not running academic backtests
   - Retail investors typically do not hold stocks through bankruptcy
   - Maintaining data on failed companies costs money and generates little traffic

4. **Data Retention Costs**
   - Storing Lehman Brothers data generates almost no revenue
   - Few users search for prices of completely delisted tickers
   - Free services reduce costs by removing unused data
   - CRSP charges universities because it maintains this long history

**Bottom Line:** Free sources optimize for current data. Historical research data typically requires paid academic databases.

---

## Q4: Why does Compustat not have detailed daily price data like CRSP?

### Answer

Compustat's core mission is **fundamental analysis** (financial statements), not price and return research.

**Key Reasons:**

1. **Different Core Purpose**
   - **Compustat:** balance sheets, income statements, cash flows, and ratios
   - **CRSP:** prices, returns, trading volume, and corporate actions
   - These support different research questions

2. **Historical Origins**
   - Compustat began as a fundamental data provider in 1962
   - Initially focused on quarterly and annual financials
   - Daily prices (comp.secd) were added later as supplementary data
   - CRSP was designed for price and return research from the beginning

3. **Market Segmentation**
   - CRSP dominates academic price and return research
   - Compustat dominates fundamental analysis research
   - They are complements, not direct competitors
   - Most academic papers use both: CRSP prices plus Compustat fundamentals

4. **Missing Fields Explained**
   - **OPENPRC** (open price): not necessary for fundamental analysis
   - **DLRET** (delisting return): not calculated by Compustat
   - **RET** (daily return): can be calculated from prccd if needed
   - Compustat provides what fundamental researchers need, not full market microstructure

5. **Research Workflow**
   - Typical academic study:
     - Obtain prices and returns from **CRSP**
     - Obtain financial ratios and statements from **Compustat**
     - Merge using a PERMNO-GVKEY link table
   - Researchers do not expect Compustat to duplicate CRSP

**Bottom Line:** Compustat and CRSP divide the market: fundamentals versus prices. Researchers use both together.

---

## Summary Table

| Feature | CRSP | Yahoo Finance | Compustat | Why the Difference? |
|---------|------|---------------|-----------|---------------------|
| Intraday prices | No | Yes (delayed) | No | Yahoo: retail traders; CRSP/Compustat: academic focus |
| Analyst recommendations | No | Yes | No | Yahoo: user engagement; others: not research relevant |
| Institutional holdings | No | Yes | No | Yahoo: retail interest; others: not historical research data |
| PERMNO (permanent ID) | Yes | No | No | CRSP invention for research; retail traders do not need it |
| GVKEY (permanent ID) | No | No | Yes | Compustat invention for fundamentals linking |
| Delisting returns (DLRET) | Yes | No | No | CRSP: research accuracy; others: no use case |
| Daily OHLC prices | Yes | Yes | Partial | CRSP: complete; Yahoo: current only; Compustat: supplementary |
| Fundamental data | No | Yes (via API) | Yes | Compustat core; CRSP not their mission |
| Historical data for delisted | Yes | No | Partial | CRSP: academic requirement; Yahoo: cost with no benefit |
| Data back to 1926 | Yes | No | No | CRSP: research database; others: modern focus |

---

## Key Takeaways

1. **Business Model Drives Features**
   - Free (Yahoo): Features that drive ad revenue (current data, recommendations)
   - Paid academic (CRSP/Compustat): Features that enable research (PERMNO, DLRET)

2. **Target Audience Matters**
   - Retail investors: Need current info, not historical rigor
   - Academic researchers: Need historical accuracy, not trading features

3. **Core Competency Focus**
   - CRSP: Price and return research
   - Compustat: Fundamental analysis
   - Yahoo: Current market information for retail

4. **They Are Complementary, Not Competing**
   - Researchers use CRSP and Compustat together
   - Retail investors use Yahoo for free current data
   - Each serves its purpose well

---

**Document Created:** December 2025  
**Purpose:** FE511 Project Part I - Data Provider Comparison
"""

# Save the markdown document
with open('/home/aditya/FE511_Project/Part_I_Why_Questions_Explained.md', 'w') as f:
    f.write(why_questions_doc)

print("Comprehensive 'Why Questions' document created")
print("  Saved to: /home/aditya/FE511_Project/Part_I_Why_Questions_Explained.md")
print("\nThis document explains:")
print("  - Why CRSP lacks intraday data")
print("  - Why Yahoo has analyst recommendations")
print("  - Why Yahoo lacks PERMNO/DLRET")
print("  - Why Compustat has limited daily price data")
print("\nYou can include this markdown file in your report or convert to PDF")

In [ ]:
# Part 1 - Cell 3.1 Visualization 1: Feature Availability Heatmap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("="*80)
print("SECTION 3.1 VISUALIZATION 1: FEATURE AVAILABILITY HEATMAP")
print("="*80 + "\n")

# Create feature availability matrix
features = [
    'Daily OHLC Prices', 
    'Adjusted Close', 
    'Trading Volume', 
    'Daily Returns (RET)', 
    'Return ex-Dividend (RETX)',
    'Delisting Returns (DLRET)', 
    'Open Price (OPENPRC)',
    'Permanent Security ID', 
    'Permanent Company ID',
    'Corporate Action Adjustments', 
    'Fundamental Data (Annual)', 
    'Fundamental Data (Quarterly)',
    'Analyst Recommendations',
    'Institutional Holdings', 
    'Earnings Estimates',
    'Intraday Data (1-min)',
    'Historical Data (20+ years)', 
    'Delisted Company Data',
    'Dividend History',
    'Stock Split History'
]

providers = ['Yahoo\nFinance', 'CRSP', 'Compustat']

# 1 = Full, 0.5 = Partial, 0 = None
availability = np.array([
    [1, 1, 0.5],      # Daily OHLC
    [1, 1, 1],        # Adjusted Close
    [1, 1, 1],        # Trading Volume
    [0, 1, 0],        # Daily Returns (RET)
    [0, 1, 0],        # Return ex-Dividend
    [0, 1, 0],        # Delisting Returns
    [1, 1, 0],        # Open Price
    [0, 1, 0],        # PERMNO
    [0, 1, 1],        # PERMCO/GVKEY
    [0.5, 1, 1],      # Corporate Actions
    [1, 0, 1],        # Fundamental Annual
    [1, 0, 1],        # Fundamental Quarterly
    [1, 0, 0],        # Analyst Recs
    [1, 0, 0],        # Institutional Holdings
    [1, 0, 0],        # Earnings Estimates
    [0.5, 0, 0],      # Intraday (delayed)
    [0.5, 1, 1],      # Historical Data
    [0, 1, 0.5],      # Delisted Companies
    [1, 1, 1],        # Dividend History
    [1, 1, 1]         # Split History
])

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(availability, annot=True, fmt='.1f', cmap='RdYlGn', 
            xticklabels=providers, yticklabels=features,
            cbar_kws={'label': 'Availability Score'},
            vmin=0, vmax=1,
            linewidths=0.5, linecolor='gray', ax=ax)

ax.set_title('Section 3.1: Feature Availability Across Data Providers\nComprehensive Feature Comparison', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Data Provider', fontsize=12, fontweight='bold')
ax.set_ylabel('Feature', fontsize=12, fontweight='bold')

# Add legend explanation
legend_text = "Score: 1.0 = Fully Available | 0.5 = Partially Available | 0.0 = Not Available"
plt.figtext(0.5, 0.01, legend_text, ha='center', fontsize=10, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('/home/aditya/FE511_Project/section_3_1_feature_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("Heatmap saved to: /home/aditya/FE511_Project/section_3_1_feature_heatmap.png")
print("\nKey Findings (Section 3.1):")
print("  1. CRSP Strengths:")
print("     - Comprehensive price/return data (RET, RETX, DLRET)")
print("     - Permanent identifiers (PERMNO, PERMCO)")
print("     - Complete historical data back to 1926")
print("  2. Yahoo Finance Strengths:")
print("     - Retail-oriented features (analyst recs, holdings, estimates)")
print("     - Delayed intraday data")
print("     - Easy API access")
print("  3. Compustat Strengths:")
print("     - Comprehensive fundamental data")
print("     - Global company identifier (GVKEY)")
print("     - Quarterly and annual financial statements")
print("\n")

In [ ]:
# Part 1 - Cell 3.1 Visualization 2: Unique vs Common Features Breakdown
print("="*80)
print("SECTION 3.1 VISUALIZATION 2: UNIQUE VS COMMON FEATURES")
print("="*80 + "\n")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Feature categories by provider
categories = ['Price/Volume\nData', 'Return\nCalculations', 'Identifiers', 
              'Fundamental\nData', 'Retail\nFeatures', 'Historical\nCompleteness']

yahoo_scores = [4, 0, 0, 4, 5, 2]  # Out of 5
crsp_scores = [5, 5, 5, 0, 0, 5]
compustat_scores = [3, 0, 3, 5, 0, 5]

x = np.arange(len(categories))
width = 0.25

bars1 = ax1.bar(x - width, yahoo_scores, width, label='Yahoo Finance', 
                color='#1f77b4', alpha=0.8, edgecolor='black')
bars2 = ax1.bar(x, crsp_scores, width, label='CRSP', 
                color='#ff7f0e', alpha=0.8, edgecolor='black')
bars3 = ax1.bar(x + width, compustat_scores, width, label='Compustat', 
                color='#2ca02c', alpha=0.8, edgecolor='black')

ax1.set_xlabel('Feature Category', fontsize=12, fontweight='bold')
ax1.set_ylabel('Capability Score (0-5)', fontsize=12, fontweight='bold')
ax1.set_title('Feature Strength by Category', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(categories, fontsize=10)
ax1.legend(fontsize=11)
ax1.set_ylim(0, 6)
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                     f'{int(height)}',
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

# Right plot: Venn diagram style - Feature overlap
from matplotlib.patches import Circle
import matplotlib.patches as mpatches

ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_aspect('equal')

# Draw circles for each provider
circle1 = Circle((3, 5), 2.5, color='#1f77b4', alpha=0.3, label='Yahoo Finance')
circle2 = Circle((7, 5), 2.5, color='#ff7f0e', alpha=0.3, label='CRSP')
circle3 = Circle((5, 3), 2.5, color='#2ca02c', alpha=0.3, label='Compustat')

ax2.add_patch(circle1)
ax2.add_patch(circle2)
ax2.add_patch(circle3)

# Add labels
ax2.text(3, 5, 'Yahoo\nFinance\nOnly:\n\n• Analyst Recs\n• Holdings\n• Estimates', 
         ha='center', va='center', fontsize=9, fontweight='bold')
ax2.text(7, 5, 'CRSP\nOnly:\n\n• PERMNO\n• RET/RETX\n• DLRET', 
         ha='center', va='center', fontsize=9, fontweight='bold')
ax2.text(5, 2.2, 'Compustat\nOnly:\n\n• GVKEY\n• Fundamentals\n• Ratios', 
         ha='center', va='center', fontsize=9, fontweight='bold')
ax2.text(5, 5.5, 'Common:\nOHLC\nVolume\nDividends', 
         ha='center', va='center', fontsize=9, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

ax2.set_title('Feature Overlap Analysis', fontsize=14, fontweight='bold')
ax2.axis('off')
ax2.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.savefig('/home/aditya/FE511_Project/section_3_1_feature_breakdown.png', dpi=300, bbox_inches='tight')
plt.show()

print("Chart saved to: /home/aditya/FE511_Project/section_3_1_feature_breakdown.png")
print("\nInterpretation:")
print("  - Each provider has unique strengths in different categories")
print("  - Common features: OHLC prices, volume, dividends, splits")
print("  - No single provider has everything - researchers often combine sources")
print("  - Yahoo: Best for retail investor needs")
print("  - CRSP: Best for price/return research")
print("  - Compustat: Best for fundamental analysis")
print("\n")

In [ ]:
# Part 1 - Cell 3.1 Visualization 3: Feature Comparison Table as Chart
print("="*80)
print("SECTION 3.1 VISUALIZATION 3: DETAILED FEATURE COMPARISON MATRIX")
print("="*80 + "\n")

fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('tight')
ax.axis('off')

# Create comprehensive feature comparison table
table_data = [
    ['Feature', 'Yahoo Finance', 'CRSP', 'Compustat', 'Primary Use Case'],
    ['Open Price', 'Yes', 'Yes (OPENPRC)', 'No', 'Intraday analysis'],
    ['High Price', 'Yes', 'Yes (ASKHI)', 'Yes (prchd)', 'Volatility measurement'],
    ['Low Price', 'Yes', 'Yes (BIDLO)', 'Yes (prcld)', 'Volatility measurement'],
    ['Close Price', 'Yes', 'Yes (PRC)', 'Yes (prccd)', 'EOD pricing'],
    ['Adjusted Close', 'Yes', 'Via CFACPR', 'Via ajexdi', 'Return calculation'],
    ['Volume', 'Yes', 'Yes (VOL)', 'Yes (cshtrd)', 'Liquidity analysis'],
    ['Daily Return', 'Calculate', 'Yes (RET)', 'Calculate', 'Performance analysis'],
    ['Return ex-Div', 'No', 'Yes (RETX)', 'No', 'Price appreciation only'],
    ['Delisting Return', 'No', 'Yes (DLRET)', 'No', 'Backtest accuracy'],
    ['Shares Outstanding', 'Via API', 'Yes (SHROUT)', 'Yes (csho)', 'Market cap calc'],
    ['PERMNO', 'No', 'Yes', 'Link table', 'Permanent security ID'],
    ['GVKEY', 'No', 'Link table', 'Yes', 'Permanent company ID'],
    ['Sales/Revenue', 'Via API', 'No', 'Yes (sale)', 'Fundamental analysis'],
    ['Total Assets', 'Via API', 'No', 'Yes (at)', 'Balance sheet'],
    ['Net Income', 'Via API', 'No', 'Yes (ni)', 'Profitability'],
    ['Analyst Ratings', 'Yes', 'No', 'No', 'Retail trading'],
    ['Inst. Holdings', 'Yes', 'No', 'No', 'Investor analysis'],
]

# Color code cells
cell_colors = []
for row in table_data:
    row_colors = []
    for i, cell in enumerate(row):
        if i == 0 or cell == 'Primary Use Case':  # Header row or column
            row_colors.append('#E0E0E0')
        elif 'Yes' in str(cell):
            row_colors.append('#90EE90')  # Light green
        elif 'No' in str(cell):
            row_colors.append('#FFB6C1')  # Light red
        elif 'Via' in str(cell) or 'Calculate' in str(cell) or 'Link' in str(cell):
            row_colors.append('#FFFACD')  # Light yellow
        else:
            row_colors.append('white')
    cell_colors.append(row_colors)

table = ax.table(cellText=table_data, cellColours=cell_colors,
                 cellLoc='left', loc='center',
                 colWidths=[0.15, 0.15, 0.15, 0.15, 0.25])

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

# Style header row
for i in range(5):
    cell = table[(0, i)]
    cell.set_facecolor('#404040')
    cell.set_text_props(weight='bold', color='white')

ax.set_title('Section 3.1: Comprehensive Feature Comparison Matrix\nGreen=Available | Red=Not Available | Yellow=Partial/Indirect', 
             fontsize=14, fontweight='bold', pad=20)

plt.savefig('/home/aditya/FE511_Project/section_3_1_feature_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Feature matrix saved to: /home/aditya/FE511_Project/section_3_1_feature_matrix.png")
print("\nThis table provides a detailed reference for Part 1, Section 3.1")
print("\n")

In [ ]:
# db.close()